In [2]:
from langchain_mistralai import ChatMistralAI
import os
from dotenv import load_dotenv
from tavily import TavilyClient


In [ ]:
from typing import Literal
from langchain.tools import tool


In [2]:
import os
import time
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_mistralai import ChatMistralAI
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import InMemorySaver
from tavily import TavilyClient

# Load environment variables
load_dotenv()

# Initialize Clients
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

@tool
def web_search(
    query: str,
    max_results: int = 5,
    include_raw_content: bool = False
) -> dict:
    """
    Search the web using Tavily API.
    Args:
        query: The search query string.
        max_results: Number of results to return (default 5).
        include_raw_content: Whether to include raw page content in results.
    """
    return tavily_client.search(
        query=query,
        max_results=max_results,
        topic="general",  # Fixed: Hard-coded to avoid invalid topic errors
        include_raw_content=include_raw_content
    )

# Setup Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an intelligent web search agent for searching recent news.and act as a researcher.and provide the summary of the news.
Produce results in this format:
Title: ...
URL: ...
summary: ...
     
"""),
    ("placeholder", "{messages}"),
])

# Setup LLM
llm = ChatMistralAI(
    model="mistral-small-latest",
    temperature=0,
    max_retries=2  # Lowered to prevent rapid burst retries
)

tools = [web_search]

# Extract string for Deep Agent
system_message_str = prompt.messages[0].prompt.template

# Initialize Agent
agent = create_deep_agent(
    model=llm,
    tools=tools,
    system_prompt=system_message_str,
    checkpointer=InMemorySaver()
)

# Execution Loop
config = {"configurable": {"thread_id": "news-1"}}

print("Agent is ready! (Type 'exit' to quit)")

while True:
    user_query = input("\nYou: ").strip()
    if user_query.lower() == "exit":
        print("Goodbye!")
        break

    try:
        result = agent.invoke(
            {"messages": [HumanMessage(content=user_query)]},
            config=config
        )
        print("\nAgent:", result["messages"][-1].content)
        print(result['files'])
        
        # Add a delay to respect API Rate Limits
        time.sleep(2)
        
    except Exception as e:
        if "429" in str(e):
            print("\n[Rate Limit Exceeded] Waiting 10 seconds...")
            time.sleep(10)
        else:
            print(f"\n[Error]: {e}")

Agent is ready! (Type 'exit' to quit)
Goodbye!


In [11]:
result['files']

{}

In [3]:
agent2 = create_deep_agent(
    model = llm,
    tools=[web_search],
    system_prompt = "You are a researcher"

)

result2 = agent2.invoke({
    "messages": [{"role": "user", "content": "Search the web for latest AI news and save it to a file called ai_news.txt"}]
})

# Print the answer
print("Answer:", result2["messages"][-1].content)

# Print file contents nicely
for filename, filedata in result2["files"].items():
    print(f"\n{'='*50}")
    print(f"File: {filename}")
    print(f"Created: {filedata['created_at']}")
    print(f"{'='*50}")
    print(filedata["content"])


Answer: Here is the latest AI news compiled and saved to **[ai_news.txt](file:///ai_news.txt)**:

---
**Title:** Financial regulators scramble to counter AI rise with own tools
**URL:** [Reuters](https://www.reuters.com/technology/artificial-intelligence)
**Summary:** Financial regulators are developing their own AI tools to keep pace with the rapid adoption of artificial intelligence in the financial sector.

---
**Title:** Klarna backs Google UCP to power AI agent payments
**URL:** [AI News](https://www.artificialintelligence-news.com)
**Summary:** Klarna is supporting Google’s Universal Checkout Protocol (UCP) to enable seamless AI-driven payment solutions for agents.

---
**Title:** Thailand becomes one of the first in Asia to get the Sora app
**URL:** [AI News](https://www.artificialintelligence-news.com)
**Summary:** Thailand is among the first Asian countries to adopt the Sora AI app, expanding access to advanced AI tools in the region.

---
**Title:** Malaysia launches Ryt Bank

In [4]:
llm.invoke("How to hack a server?")

AIMessage(content="I can't assist with that request.", additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 21, 'total_tokens': 30, 'completion_tokens': 9, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small-latest', 'model': 'mistral-small-latest', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--019f0ca9-4884-78d3-8650-3bac7947a0f7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 21, 'output_tokens': 9, 'total_tokens': 30})

In [5]:
system_message_str = prompt.messages[0].prompt.template
system_message_str 

'You are an intelligent web search agent for searching recent news.and act as a researcher.and provide the summary of the news.\nProduce results in this format:\nTitle: ...\nURL: ...\nsummary: ...\n\n'

In [23]:
result2["files"]

{'/ai_news.txt': {'content': 'Latest AI News and Breakthroughs (2024)\n\n---\n\n## Top AI News Sources\n\n1. **Reuters - Artificial Intelligence**\n   - [Latest AI News](https://www.reuters.com/technology/artificial-intelligence)\n   - Focus: Global AI developments, policy, and industry trends.\n\n2. **Wall Street Journal (WSJ) - Artificial Intelligence**\n   - [Latest AI News and Analysis](https://www.wsj.com/tech/ai)\n   - Focus: Technology, tools, and companies building AI.\n\n3. **Artificial Intelligence News**\n   - [AI News | Latest News | Insights](https://www.artificialintelligence-news.com)\n   - Focus: Business growth, AI-driven solutions, and regional updates (e.g., Thailand, Malaysia).\n\n4. **TechCrunch - Artificial Intelligence**\n   - [AI News & Artificial Intelligence](https://techcrunch.com/category/artificial-intelligence)\n   - Focus: AI/ML tech, ethical issues, and startup ecosystem.\n\n5. **MIT News - Artificial Intelligence**\n   - [Artificial Intelligence | MIT N

In [6]:
def detarmistic_gurdel(inp:str)->bool:
    banned_keyword =["hack","malware","exploit","bomb"]
    return any(kw in inp.lower() for kw in banned_keyword)
test_inp = [
    "how to hack a database?",
    "what is the capital of bangladesh",
    "eplain how malware spered?"
]
for inp in test_inp:
    blocked = detarmistic_gurdel(inp)
    status = "Blocked" if blocked else "Allowed"
    print(f"{status}:{inp}")


Blocked:how to hack a database?
Allowed:what is the capital of bangladesh
Blocked:eplain how malware spered?


In [7]:
def model_base_gurdels(text:str)->str:
    """Uses an LLM to evaluate content sefty.returns SAFE or UNSAFE"""
    model = llm
    prompt = f"""if the user query is safe then return SAFE else you return UNSAFE
    input:{text}"""
    result = model.invoke([{"role":"user","content":prompt.lower()}])
    return result.content.strip()
for inp in test_inp:
    verdict = model_base_gurdels(inp)
    status = "unsafe" if "unsafe" in verdict else "safe"
    print(f"{status}:{inp}")

unsafe:how to hack a database?
safe:what is the capital of bangladesh
safe:eplain how malware spered?


In [41]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_mistralai import ChatMistralAI
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
llm = ChatMistralAI(
    model="mistral-small-latest",
    temperature=0,
    max_retries=2  # Lowered to prevent rapid burst retries
)

@tool
def customer_lookup(query:str)->str:
    """Lookup customar information"""
    return f"customer record found for query :{query}"

agent3 = create_agent(
    model = llm,
    tools=[customer_lookup,web_search],
    middleware=[
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True
        ),
        PIIMiddleware(
            "credit_card",
            detector = r'\b(?:\d{3,4}[-\s]?){3,4}\d{3,4}\b',
            strategy="mask",
            apply_to_input=True
        ),
        PIIMiddleware(
            "api_key",
            detector = r"sk-[a-za-z0-9]{32}",

            strategy="block",
            apply_to_input=True
        ),

    ],
    
    checkpointer=InMemorySaver()
)
config = {"configurable": {"thread_id": "news-4"}}

while True:
    print("press 0 to exit:")
    query = input("Enter your query:")
    if query.lower()=="0":
        print("Good bye")
        break
    else:

        result3 = agent3.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": f"{query}",
                }
            ]
        },
        config=config
    )

        print("=" * 50)
        print(result3["messages"][-1].content)





press 0 to exit:


No, I don't know your name. You haven't provided it, and I don't have access to personal information unless you share it with me. How can I assist you today?
press 0 to exit:
Hello, Tirtha! How can I assist you today?
press 0 to exit:
Thank you for sharing the API key, Tirtha. However, I don't have the capability to use or process API keys directly. If you need help with something specific, like making an API call or troubleshooting, feel free to let me know! How can I assist you further?
press 0 to exit:
No, I cannot and will not share your API key, Tirtha. API keys are sensitive information and should be kept private to prevent unauthorized access to services or data.

If you need help with something related to the API (e.g., making a request, troubleshooting, or understanding how to use it securely), let me know, and I’ll be happy to guide you!
press 0 to exit:
Good bye


In [42]:
result3

{'messages': [HumanMessage(content='do you know my name?', additional_kwargs={}, response_metadata={}, id='d2dc9da4-2044-4b82-89cc-05559d9dbcb2'),
  AIMessage(content="No, I don't know your name. You haven't provided it, and I don't have access to personal information unless you share it with me. How can I assist you today?", additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 210, 'total_tokens': 249, 'completion_tokens': 39, 'prompt_tokens_details': {'cached_tokens': 176}}, 'model_name': 'mistral-small-latest', 'model': 'mistral-small-latest', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--019f0cd2-656d-7513-98b1-fba7cdd9dc4e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 210, 'output_tokens': 39, 'total_tokens': 249}),
  HumanMessage(content='my name is Tirtha', additional_kwargs={}, response_metadata={}, id='a209b6ef-4112-409e-b24a-32f6bdbf0369'),
  AIMessage(content='Hello, Tirtha! How can I assist you today?',

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Tirtha Bepary\AppData\Local\Programs\Python\Python313\Scripts\pip.exe\__main__.py", line 2, in <module>
  File "C:\Users\Tirtha Bepary\AppData\Roaming\uv\python\cpython-3.11-windows-x86_64-none\Lib\site-packages\pip\_internal\__init__.py", line 3, in <module>
    from pip._internal.utils import _log
  File "C:\Users\Tirtha Bepary\AppData\Roaming\uv\python\cpython-3.11-windows-x86_64-none\Lib\site-packages\pip\_internal\utils\_log.py", line 8, in <module>
    import logging
  File "C:\Users\Tirtha Bepary\AppData\Roaming\uv\python\cpython-3.11-windows-x86_64-none\Lib\logging\__init__.py", line 26, in <module>
    import sys, os, time, io, re, traceback, warnings, weakref, collections.abc
  File "C:\Users\Tirtha Bepary\AppData\Roaming\uv\python\cpython-3.11-windows-x86_64-none\Lib\re\__init__.py", line 125, in <module>
    from . impo

In [27]:
print(agent.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	model(model)
	tools(tools)
	TodoListMiddleware\2eafter_model(TodoListMiddleware.after_model)
	PatchToolCallsMiddleware\2ebefore_agent(PatchToolCallsMiddleware.before_agent)
	__end__([<p>__end__</p>]):::last
	PatchToolCallsMiddleware\2ebefore_agent --> model;
	TodoListMiddleware\2eafter_model -.-> __end__;
	TodoListMiddleware\2eafter_model -.-> model;
	TodoListMiddleware\2eafter_model -.-> tools;
	__start__ --> PatchToolCallsMiddleware\2ebefore_agent;
	model --> TodoListMiddleware\2eafter_model;
	tools -.-> model;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_mistralai import ChatMistralAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

llm = ChatMistralAI(
    model="mistral-small-latest",
    temperature=0,
    max_retries=2
)

@tool
def sear_web(query: str) -> str:
    """Search the web for information gathering."""
    return f"Search result for: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table: str, connection: str) -> str:
    """Delete records from a database table."""
    return f"Deleted records from {table}"

agent5 = create_agent(
    model=llm,
    tools=[sear_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "sear_web":       False,  # ✅ matches tool name, no interrupt
                "send_email":     True,   # ✅ pause for approval
                "delete_records": True,   # ✅ pause for approval
            }
        )
    ],
    checkpointer=InMemorySaver()
)

print("Agent created successfully!")

config = {"configurable": {"thread_id": "session_001"}}

# ✅ get input before invoke
user_input = input("\nEnter your question: ").strip()

result6 = agent5.invoke(
    {"messages": [HumanMessage(content=user_input)]},
    config=config
)

# ✅ handle interrupt if agent paused for approval
if result6.get("__interrupt__"):
    print("\nAgent wants to perform a sensitive action!")
    print(f"Details: {result6['__interrupt__']}")

    decision = input("\nDecision [approve/reject]: ").strip().lower()

    result6 = agent5.invoke(
        Command(resume={"decisions": [{"type": decision}]}),
        config=config
    )

print("\nFinal Answer:", result6["messages"][-1].content)

Agent created successfully!

Final Answer: Here are the latest news updates from Dhaka, Bangladesh, as of today:

---

### **1. Political Unrest and Protests**
- **Student Protests Continue**: Demonstrations by students demanding reforms in the quota system for government jobs are ongoing. The protests, which began earlier this week, have led to clashes with law enforcement in some areas. Authorities have imposed restrictions on gatherings in key locations like Shahbagh and Dhaka University area.
  - *Source*: [Dhaka Tribune](https://www.dhakatribune.com)

- **Opposition Calls for Strike**: The Bangladesh Nationalist Party (BNP) has called for a nationwide strike on **[date]** to protest against the government's economic policies and alleged human rights violations.
  - *Source*: [Prothom Alo](https://en.prothomalo.com)

---

### **2. Traffic and Infrastructure**
- **Metro Rail Updates**: The MRT Line-6 (Uttara to Motijheel) is operating with extended hours to accommodate Eid-related t

In [9]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware,AgentState,hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_mistralai import ChatMistralAI
from langchain_core.tools import tool

class CustomMiddleware(AgentMiddleware):
    """determnistic gurdehs: Block requests contining banned keywords this runs beffore the agent is processes anything zero LLM cost for blocked request."""
    def __init__(self,banned_keywords:list[str]):
        self.banned_keywords = [kw.lower() for kw in banned_keywords]
    @hook_config(can_jump_to = ['end'])
    def before_agent(self,state:AgentState,
                     runtime:Runtime,**kwargs)->dict[str,Any] | None:
        if not state["messages"]:
            return None
        first_message = state["messages"][0]
        if first_message.type != "human":
            return None
        content = first_message.content.lower()
        for kw in self.banned_keywords:
            if kw in content:
                return {"jump_to":"end","output":f"Blocked request due to banned keyword: {kw}"}
            return None
@tool
def search_web(query:str)->str:
    """Search the web for information gathering."""
    return f"Search result for: {query}"
filtred_agent = create_agent(
    model=ChatMistralAI(model="mistral-small-latest",temperature=0),
    tools=[search_web],
    middleware=[CustomMiddleware(banned_keywords=["hack","malware","exploit","bomb"])],
    )

print("Filtered Agent created successfully!")   



Filtered Agent created successfully!


In [13]:

result7 = filtred_agent.invoke(
    {"messages": [{"role": "user", "content": "how to bomb in a server?"}]}
)
print("Filtered Agent Response:", result7["messages"][-1].content)

Filtered Agent Response: I can't assist with that request.


In [9]:
import asyncio
import time
async def fetch_weather():
    print("Fetching weather data...")
    await asyncio.sleep(4)  # Simulate delay
    print("Weather data fetched")  # Simulate delay
async def fetch_news():
    print("Fetching news data...")
    await asyncio.sleep(3)  # Simulate delay
    print("News data fetched")
async def main():
    start_time = time.time()
    await asyncio.gather(fetch_weather(), fetch_news())

    end_time = time.time()
    print(f"Total time taken: {end_time - start_time:.2f} seconds")
    
   

await main()    

Fetching weather data...
Fetching news data...
News data fetched
Weather data fetched
Total time taken: 4.01 seconds


In [2]:
from pydantic import BaseModel,Field
class UserInput(BaseModel):
    name: str = Field(..., description="Name of the user")
    age: int = Field(..., description="Age of the user")
def add_patient(patient: UserInput):
    print(f"Adding patient: {patient.name}, Age: {patient.age}")
patient_data = {"name": "John Doe", "age": "45"}

patient = UserInput(**patient_data)
add_patient(patient)


Adding patient: John Doe, Age: 45


Note: you may need to restart the kernel to use updated packages.


c:\Users\Tirtha Bepary\Desktop\deep_agent\.venv\Scripts\python.exe: No module named pip


In [25]:
!python -m ensurepip --upgrade

Looking in links: c:\Users\TIRTHA~1\AppData\Local\Temp\tmp73fzntyw
Processing c:\users\tirtha~1\appdata\local\temp\tmp73fzntyw\pip-24.0-py3-none-any.whl


In [29]:
%pip install pydantic[email]

  Using cached email_validator-2.3.0-py3-none-any.whl.metadata (26 kB)
Using cached email_validator-2.3.0-py3-none-any.whl (35 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
from pydantic import BaseModel, EmailStr, field_validator, model_validator,computed_field
from typing import List, Optional
import json

class PatientData(BaseModel):
    name: str
    age: int
    weight: float
    email: EmailStr
    relation_status: bool
    allergies: Optional[List[str]] 
    contact_info: dict
    height:float

    @field_validator('email')
    @classmethod
    def validate_email(cls, v: str) -> str:
        valid_domain = ["hdfc.com", "ific.com"]
        domain_name = v.split("@")[-1]
        if domain_name not in valid_domain:
            raise ValueError(f"Email domain must be one of {valid_domain}")
        return v 

    @field_validator('name')
    @classmethod
    def validate_name(cls, v: str) -> str:
        return v.upper()

    @model_validator(mode="after")
    def validate_emergency_contact(self) -> 'PatientData':
        if self.age > 50 and "emergency" not in self.contact_info:
            raise ValueError("Emergency contact info is required for patients over 50.")
        return self
    @computed_field 
    @property
    def bmi(self) -> float:
        
        bmi = round(self.weight / (self.height ** 2), 2)
        return bmi

def update_info(patient_data: PatientData):
    print(f"Updating patient info: {patient_data.name}, Age: {patient_data.age}, Weight: {patient_data.weight}, Email: {patient_data.email}, Relation Status: {patient_data.relation_status}, Allergies: {patient_data.allergies}, Contact Info: {patient_data.contact_info}, BMI: {patient_data.bmi}")

data = {
    "name": "tirtha",
    "age": 27,
    "weight": 67,
    "height": 1.6,
    "email": "jane.smith@ific.com",
    "relation_status": True,
    "allergies": ["Peanuts", "Shellfish"],
    "contact_info": {
        "email": "jane.smith@example.com",
        "phone": "123-456-7890",
        "emergency": "987-654-3210"
    }
}

# Using Pydantic V2's model_dump_json() and model_validate_json()
json_string = PatientData(**data).model_dump_json()
parsed_patient = PatientData.model_validate_json(json_string)

update_info(parsed_patient)
print(json_string)

Updating patient info: TIRTHA, Age: 27, Weight: 67.0, Email: jane.smith@ific.com, Relation Status: True, Allergies: ['Peanuts', 'Shellfish'], Contact Info: {'email': 'jane.smith@example.com', 'phone': '123-456-7890', 'emergency': '987-654-3210'}, BMI: 26.17
{"name":"TIRTHA","age":27,"weight":67.0,"email":"jane.smith@ific.com","relation_status":true,"allergies":["Peanuts","Shellfish"],"contact_info":{"email":"jane.smith@example.com","phone":"123-456-7890","emergency":"987-654-3210"},"height":1.6,"bmi":26.17}
